# Tutorial · Emulasi Jaringan dengan **Mininet** (tanpa controller Ryu)

Tutorial ini mengajarkan **Mininet sebagai emulator jaringan**: membangun bermacam topologi, mengatur parameter link, mengukur kinerja, menguji ketahanan, lalu **menyiapkan dataset** untuk dua tujuan:

- **optimasi jaringan** → berkas topologi JSON (format sama dengan `data/topology.json`) yang langsung dapat dimuat oleh `01_optimasi_jaringan.ipynb` dan notebook Colab, plus matriks RTT terukur;
- **deteksi DDoS** → rekaman trafik di server untuk dijadikan fitur per jendela waktu.

Controller Ryu dibahas di tutorial terpisah. Di sini Open vSwitch berjalan **tanpa controller** memakai dua mode:

| Mode | Cara kerja | Untuk topologi |
|---|---|---|
| `standalone` | OVS sebagai switch L2 biasa (MAC learning bawaan) | tanpa loop: single, linear, tree |
| `rstp` | standalone + Rapid Spanning Tree memblokir port pembentuk loop | ber-loop: ring, mesh, fat-tree |

> Jalankan Jupyter sebagai root: `bash run_jupyter.sh` (Linux) atau `run_jupyter.bat wsl` (Windows). Di luar itu, pakai tutorial NetworkX di folder `colab/`.

## 0. Konsep Mininet
- **Host** = proses pada *network namespace* tersendiri (punya interface, IP, dan tabel routing sendiri).
- **Switch** = bridge Open vSwitch.
- **Link** = pasangan *veth*; `TCLink` menambah `tc` untuk membatasi bandwidth (HTB), menambah delay, dan loss (netem).
- Menjalankan perintah pada host: `net.get('h1').cmd('ping -c3 10.0.0.2')`.

Padanan baris perintah untuk sebagian isi notebook ini:
`sudo mn --topo tree,depth=2,fanout=2 --link tc,bw=10,delay=5ms --switch ovsbr --controller none`

In [ ]:
# ============================ KONFIGURASI ============================
TOPOLOGI_UJI = ["single", "linear", "tree", "ring", "mesh", "fat-tree", "json"]
PING_COUNT = 5
IPERF_SECONDS = 3
SEED = 42

import os, sys, json, time, math, re, shutil, platform, subprocess, itertools
from collections import defaultdict
from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = BASE / "results" / "tutorial_mininet"
DATA = BASE / "data"
for d in (OUT, DATA):
    d.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)

if platform.system() != "Linux" or os.geteuid() != 0:
    raise SystemExit("Butuh Linux/WSL2 + Jupyter sebagai root (bash run_jupyter.sh). "
                     "Alternatif: colab/tutorial_simulasi_jaringan_networkx.ipynb")
from mininet.net import Mininet
from mininet.node import OVSSwitch, OVSBridge
from mininet.link import TCLink
from mininet.clean import cleanup
from mininet.log import setLogLevel
setLogLevel("warning")

OVS_DATAPATH = "kernel" if (os.path.isdir("/sys/module/openvswitch") or
    (shutil.which("modprobe") and subprocess.run(["modprobe", "openvswitch"], capture_output=True).returncode == 0)) else "user"
for tool in ("ovs-vsctl", "ovs-ofctl", "iperf", "tcpdump", "ethtool"):
    print(f"{'✔' if shutil.which(tool) else '✘'} {tool}")
print("Datapath OVS:", OVS_DATAPATH)

## 1. Katalog topologi (cetak biru dengan NetworkX)
Tiap topologi dibuat dulu sebagai **graf NetworkX** (node switch + host, atribut link `delay`/`bandwidth`/`loss`), lalu graf itu dibangun menjadi jaringan Mininet. Dengan begitu topologi apa pun — termasuk berkas JSON — memakai jalur kode yang sama, dan ekspor dataset menjadi mudah.

In [ ]:
def _rand_attrs(rng):
    return dict(delay=float(rng.choice([1, 2, 3, 5])), bandwidth=float(rng.choice([20, 50, 100])),
                loss=float(rng.choice([0, 0, 0.001, 0.002])))

def blueprint(kind, n=4, depth=2, fanout=2, k=4, seed=SEED, json_path=None):
    """Graf cetak biru. Node switch: type='switch'; host: type='host'. Edge membawa delay/bandwidth/loss."""
    rng = np.random.default_rng(seed)
    S = nx.Graph()
    if kind == "single":
        S.add_node("s1")
    elif kind == "linear":
        nx.add_path(S, [f"s{i+1}" for i in range(n)])
    elif kind == "tree":
        S = nx.relabel_nodes(nx.balanced_tree(fanout, depth), lambda u: f"s{u+1}")
    elif kind == "ring":
        nx.add_cycle(S, [f"s{i+1}" for i in range(n)])
    elif kind == "mesh":
        S = nx.relabel_nodes(nx.complete_graph(n), lambda u: f"s{u+1}")
    elif kind == "fat-tree":
        core = [f"s{i+1}" for i in range((k // 2) ** 2)]
        S.add_nodes_from(core); cnt = len(core)
        for _pod in range(k):
            aggs = [f"s{cnt+i+1}" for i in range(k // 2)]; cnt += k // 2
            edges = [f"s{cnt+i+1}" for i in range(k // 2)]; cnt += k // 2
            for i, a in enumerate(aggs):
                for j in range(k // 2):
                    S.add_edge(a, core[i * (k // 2) + j])
                for e in edges:
                    S.add_edge(a, e)
            for e in edges:
                S.nodes[e]["leaf"] = True
    elif kind == "json":
        data = json.loads(Path(json_path or DATA / "topology.json").read_text())
        name = {u: f"s{i+1}" for i, u in enumerate(data["nodes"])}
        for e in data["edges"]:
            S.add_edge(name[e["source"]], name[e["target"]],
                       delay=float(e["delay"]), bandwidth=float(e["bandwidth"]), loss=float(e["loss"]))
    else:
        raise ValueError(kind)
    G = nx.Graph(kind=kind)
    for u in S:
        G.add_node(u, type="switch", leaf=S.nodes[u].get("leaf", False))
    for u, v, d in S.edges(data=True):
        G.add_edge(u, v, **(d if d else _rand_attrs(rng)))
    # tempelkan host
    if kind == "single":
        attach = {"s1": n}
    elif kind == "tree":
        attach = {u: fanout for u in S if S.degree(u) == 1}
    elif kind == "fat-tree":
        attach = {u: k // 2 for u in S if S.nodes[u].get("leaf")}
    else:
        attach = {u: 1 for u in S}
    h = 0
    for sw in sorted(attach, key=lambda s: int(s[1:])):
        for _ in range(attach[sw]):
            h += 1
            G.add_node(f"h{h}", type="host", ip=f"10.0.0.{h}", mac=f"00:00:00:00:00:{h:02x}")
            G.add_edge(f"h{h}", sw, delay=0.0, bandwidth=1000.0, loss=0.0)
    return G

def switches(G):
    return sorted((u for u in G if G.nodes[u]["type"] == "switch"), key=lambda s: int(s[1:]))
def hosts(G):
    return sorted((u for u in G if G.nodes[u]["type"] == "host"), key=lambda s: int(s[1:]))

def layout(G):
    S = G.subgraph(switches(G))
    if G.graph["kind"] in ("tree", "fat-tree") and S.number_of_nodes() > 1:
        roots = [switches(G)[0]] if G.graph["kind"] == "tree" else switches(G)[:(4 // 2) ** 2 or 1]
        lv = nx.multi_source_dijkstra_path_length(S, roots)
        layers = defaultdict(list)
        for u, l in lv.items():
            layers[l].append(u)
        pos = {u: (i - (len(us) - 1) / 2, -l) for l, us in layers.items() for i, u in enumerate(sorted(us))}
    else:
        pos = nx.circular_layout(S) if G.graph["kind"] in ("ring", "mesh") else nx.kamada_kawai_layout(S) if S.number_of_edges() else {switches(G)[0]: (0, 0)}
    # letakkan host dekat switch induknya
    for hname in hosts(G):
        sw = next(G.neighbors(hname))
        base = np.array(pos[sw]); off = np.random.default_rng(int(hname[1:])).uniform(-0.25, 0.25, 2)
        pos[hname] = tuple(base + np.array([0, -0.4]) + off)
    return pos

def draw(G, ax, title=None, highlight=None):
    pos = layout(G)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#c3cbd4")
    if highlight:
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=highlight, edge_color="#e8743b", width=3)
    nx.draw_networkx_nodes(G, pos, nodelist=switches(G), ax=ax, node_color="#2f6db5", node_shape="s", node_size=170)
    nx.draw_networkx_nodes(G, pos, nodelist=hosts(G), ax=ax, node_color="#1baf7a", node_size=80)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=6, font_color="white")
    ax.set_title(title or G.graph["kind"], fontsize=10); ax.axis("off")
    return pos

PARAMS = {"single": dict(n=4), "linear": dict(n=4), "tree": dict(depth=2, fanout=2),
          "ring": dict(n=6), "mesh": dict(n=4), "fat-tree": dict(k=4), "json": {}}
BLUEPRINTS = {kind: blueprint(kind, **PARAMS[kind]) for kind in TOPOLOGI_UJI}
fig, axes = plt.subplots(1, len(BLUEPRINTS), figsize=(3.3 * len(BLUEPRINTS), 3.3))
for ax, (kind, Gk) in zip(np.atleast_1d(axes), BLUEPRINTS.items()):
    S = Gk.subgraph(switches(Gk))
    loop = "tanpa loop" if nx.is_forest(S) else "ber-loop"
    draw(Gk, ax, f"{kind}\n{len(switches(Gk))} switch · {len(hosts(Gk))} host · {loop}")
plt.tight_layout(); plt.savefig(OUT / "katalog_topologi.png", dpi=150); plt.show()

## 2. Membangun jaringan Mininet dari cetak biru
`build(G, mode)` menerjemahkan graf menjadi `OVSBridge` (tanpa controller). Untuk topologi ber-loop dipakai `mode="rstp"` sehingga Rapid Spanning Tree memblokir port yang membentuk loop; kita menunggu sampai semua host saling terhubung dan mencatat waktu konvergensinya.

> Pada datapath **userspace** (lihat sel 0), TX checksum offload interface veth dimatikan dengan `ethtool -K ... tx off`; tanpa itu paket TCP dibuang dan iperf/HTTP gagal walau ping jalan.

In [ ]:
def build(G, mode="standalone"):
    cleanup()
    net = Mininet(switch=partial(OVSBridge, datapath=OVS_DATAPATH), link=TCLink,
                  controller=None, autoSetMacs=False, build=False)
    for s in switches(G):
        net.addSwitch(s, dpid=f"{int(s[1:]):016x}")
    for h in hosts(G):
        net.addHost(h, ip=G.nodes[h]["ip"] + "/24", mac=G.nodes[h]["mac"])
    for u, v, d in G.edges(data=True):
        opts = dict(bw=d["bandwidth"])
        if d["delay"] > 0:
            opts["delay"] = f"{d['delay']}ms"
        if d["loss"] > 0:
            opts["loss"] = d["loss"] * 100
        net.addLink(u, v, **opts)
    net.build(); net.start()
    if OVS_DATAPATH == "user":
        for h in net.hosts:
            for intf in h.intfList():
                h.cmd(f"ethtool -K {intf} tx off > /dev/null 2>&1")
    if mode == "rstp":
        for s in net.switches:
            s.cmd(f"ovs-vsctl set Bridge {s.name} rstp_enable=true")
    return net

def wait_connectivity(net, timeout=60):
    """pingAll berulang sampai 0% loss; kembalikan waktu konvergensi (detik)."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        if net.pingAll(timeout=1) == 0:
            return round(time.time() - t0, 1)
        time.sleep(1)
    return float("nan")

def port_map(net):
    pm = {}
    for link in net.links:
        a, b = link.intf1, link.intf2
        pm[(a.node.name, b.node.name)] = a.node.ports[a]
        pm[(b.node.name, a.node.name)] = b.node.ports[b]
    return pm

# contoh: tree depth=2 fanout=2 dalam mode standalone
net = build(BLUEPRINTS["tree"], "standalone")
print("Konvergensi:", wait_connectivity(net), "s")
print(net.get("h1").cmd("ping -c3 " + BLUEPRINTS["tree"].nodes[hosts(BLUEPRINTS['tree'])[-1]]["ip"]))
print("Tabel MAC switch s1:\n" + net.get("s1").cmd("ovs-appctl fdb/show s1"))
net.stop()

## 3. Eksperimen di berbagai topologi
Untuk tiap topologi: bangun jaringan, tunggu konvergensi, ukur **RTT semua pasangan host** (`ping`) dan **throughput TCP** (`iperf`) antar host terjauh. Metrik graf (diameter, konektivitas) dihitung NetworkX sebagai pembanding.

In [ ]:
def rtt_matrix(net, G, count=PING_COUNT):
    rows = []
    for a, b in itertools.combinations(hosts(G), 2):
        out = net.get(a).cmd(f"ping -c {count} -i 0.2 -W 1 {G.nodes[b]['ip']}")
        m = re.search(r"= [\d.]+/([\d.]+)/([\d.]+)/", out)
        loss = re.search(r"([\d.]+)% packet loss", out)
        rows.append(dict(src=a, dst=b, rtt_ms=float(m.group(1)) if m else np.nan,
                         rtt_maks_ms=float(m.group(2)) if m else np.nan,
                         loss_pct=float(loss.group(1)) if loss else 100.0))
    return pd.DataFrame(rows)

def iperf_pair(net, G, a, b, seconds=IPERF_SECONDS):
    net.get(b).cmd("iperf -s -p 5001 > /dev/null 2>&1 &"); time.sleep(0.5)
    out = net.get(a).cmd(f"timeout {seconds + 10} iperf -c {G.nodes[b]['ip']} -p 5001 -t {seconds} -f m")
    net.get(b).cmd("pkill -f 'iperf -s -p 5001'")
    v = re.findall(r"([\d.]+) Mbits/sec", out)
    return float(v[-1]) if v else np.nan

summary, RTT = [], {}
for kind, Gk in BLUEPRINTS.items():
    S = Gk.subgraph(switches(Gk))
    mode = "standalone" if nx.is_forest(S) else "rstp"
    net = build(Gk, mode)
    conv = wait_connectivity(net)
    rtt = rtt_matrix(net, Gk); RTT[kind] = rtt
    far = max(itertools.combinations(hosts(Gk), 2), key=lambda p: nx.shortest_path_length(Gk, *p, weight="delay"))
    thr = iperf_pair(net, Gk, *far)
    net.stop()
    summary.append(dict(topologi=kind, mode=mode, switch=S.number_of_nodes(), host=len(hosts(Gk)),
                        link=S.number_of_edges(), diameter_hop=nx.diameter(S) if S.number_of_nodes() > 1 else 0,
                        konektivitas=nx.edge_connectivity(S) if S.number_of_nodes() > 1 else 0,
                        konvergensi_s=conv, rtt_rata_ms=rtt.rtt_ms.mean(), rtt_maks_ms=rtt.rtt_maks_ms.max(),
                        loss_pct=rtt.loss_pct.mean(), terjauh=f"{far[0]}-{far[1]}", iperf_mbps=thr))
    print(f"✔ {kind:<9} mode={mode:<10} konv={conv}s  RTT rata={rtt.rtt_ms.mean():.2f} ms  iperf={thr:.1f} Mbps")
summary = pd.DataFrame(summary).set_index("topologi")
summary.to_csv(OUT / "ringkasan_topologi.csv")
display(summary.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, (col, lbl) in zip(axes, [("rtt_rata_ms", "RTT rata-rata (ms)"), ("iperf_mbps", "Throughput TCP terjauh (Mbps)"),
                                 ("konvergensi_s", "Waktu konvergensi (s)")]):
    ax.bar(summary.index, summary[col], color=["#2f6db5" if m == "standalone" else "#e8743b" for m in summary["mode"]])
    ax.set_title(lbl, fontsize=10); ax.tick_params(axis="x", rotation=30)
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c, label=l) for c, l in
                        [("#2f6db5", "standalone"), ("#e8743b", "rstp (ber-loop)")]], fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "perbandingan_topologi.png", dpi=150); plt.show()

# matriks RTT salah satu topologi
kind = "json" if "json" in RTT else list(RTT)[-1]
hs = hosts(BLUEPRINTS[kind])
M = pd.DataFrame(np.nan, index=hs, columns=hs)
for x in RTT[kind].itertuples():
    M.loc[x.src, x.dst] = M.loc[x.dst, x.src] = x.rtt_ms
fig, ax = plt.subplots(figsize=(1 + 0.4 * len(hs), 0.8 + 0.4 * len(hs)))
im = ax.imshow(M.values.astype(float), cmap="viridis"); plt.colorbar(im, ax=ax, label="RTT (ms)")
ax.set_xticks(range(len(hs))); ax.set_xticklabels(hs, rotation=90, fontsize=7)
ax.set_yticks(range(len(hs))); ax.set_yticklabels(hs, fontsize=7)
ax.set_title(f"Matriks RTT terukur — {kind}")
plt.tight_layout(); plt.savefig(OUT / f"matriks_rtt_{kind}.png", dpi=150); plt.show()

## 4. Ubah parameter link & uji ketahanan (link putus)
Pada **ring** (mode RSTP): ukur RTT h1→h4, ubah delay satu link saat jaringan berjalan, lalu **putuskan** sebuah link. RSTP memindahkan jalur ke port cadangan; waktu pemulihan diukur dari ping beruntun.

In [ ]:
net = build(blueprint("ring", n=6), "rstp")
print("Konvergensi awal:", wait_connectivity(net), "s")
h1, h4 = net.get("h1"), net.get("h4")
def rtt(a, b, n=5):
    m = re.search(r"= [\d.]+/([\d.]+)/", a.cmd(f"ping -c {n} -i 0.2 -W 1 {b.IP()}"))
    return float(m.group(1)) if m else float("nan")
print(f"RTT h1→h4 awal                 : {rtt(h1, h4):.2f} ms")
link = net.linksBetween(net.get("s1"), net.get("s2"))[0]
for intf in (link.intf1, link.intf2):
    intf.config(delay="20ms", bw=50)
print(f"RTT setelah delay s1-s2 =20 ms  : {rtt(h1, h4):.2f} ms")

h1.cmd(f"ping -D -i 0.1 -W 1 {h4.IP()} > /tmp/ping_fail.log 2>&1 &")
time.sleep(3); t_fail = time.time()
net.configLinkStatus("s3", "s4", "down")
time.sleep(10); h1.cmd("pkill -f 'ping -D'")
net.stop()
ts = [float(m.group(1)) for m in re.finditer(r"\[(\d+\.\d+)\].*time=", open("/tmp/ping_fail.log").read())]
after = [t for t in ts if t > t_fail]
print(f"Pemulihan setelah s3-s4 putus  : {after[0] - t_fail:.2f} s" if after else "tidak pulih dalam 10 s")
fig, ax = plt.subplots(figsize=(10, 2.4))
ax.plot(np.array(ts) - t_fail, np.ones(len(ts)), "|", ms=16, color="#2f6db5")
ax.axvline(0, color="red", ls="--"); ax.set_yticks([]); ax.set_xlabel("detik relatif terhadap link putus")
ax.set_title("Balasan ping h1→h4 (celah = paket hilang saat RSTP konvergen ulang)")
plt.tight_layout(); plt.savefig(OUT / "uji_link_putus.png", dpi=150); plt.show()

## 5. Ekspor dataset untuk **algoritma optimasi jaringan**
Topologi diekspor sebagai JSON berformat sama dengan `data/topology.json` (plus `hosts` dan `pos`), sehingga bisa langsung dimuat oleh `01_optimasi_jaringan.ipynb` lewat `TOPOLOGY_FILE` atau oleh notebook Colab. Matriks RTT terukur disimpan berdampingan sebagai pembanding "dunia nyata" terhadap estimasi analitik.

In [ ]:
def export_topology_json(G, path, description):
    S = G.subgraph(switches(G))
    pos = layout(G)
    data = {"description": description, "nodes": switches(G),
            "edges": [dict(source=u, target=v, delay=d["delay"], bandwidth=d["bandwidth"], loss=d["loss"])
                      for u, v, d in S.edges(data=True)],
            "hosts": {h: next(G.neighbors(h)) for h in hosts(G)},
            "pos": {u: [round(float(pos[u][0]), 4), round(float(pos[u][1]), 4)] for u in switches(G)}}
    Path(path).write_text(json.dumps(data, indent=1))
    return path

for kind, Gk in BLUEPRINTS.items():
    if len(switches(Gk)) < 2:
        continue
    p = export_topology_json(Gk, DATA / f"topologi_{kind}.json",
                             f"Topologi {kind} dari tutorial Mininet (delay ms, bandwidth Mbps, loss probabilitas)")
    RTT[kind].to_csv(OUT / f"rtt_{kind}.csv", index=False)
    print("ekspor:", p.relative_to(BASE), "+", f"rtt_{kind}.csv")
print("\nUntuk memakainya di 01_optimasi_jaringan.ipynb:")
print('  TOPOLOGY_SOURCE = "file";  TOPOLOGY_FILE = "data/topologi_mesh.json"')

## 6. Rekaman trafik untuk **deteksi DDoS**
Di sini kita hanya **merekam trafik di server** dengan `tcpdump` lalu mengubahnya menjadi tabel paket + fitur per jendela waktu. Pembangkitan trafik serangan dan model deteksi sepenuhnya memakai notebook `02_deteksi_ddos.ipynb`; sel di bawah menunjukkan cara mengubah hasil `tcpdump` apa pun (termasuk dari skenario di notebook 02) menjadi dataset.

Topologi: satu server `srv` (10.0.0.100) di belakang link sempit, dengan beberapa client. `tcpdump` pada interface server menulis `pcap`; `tshark`/`tcpdump -r` mengubahnya ke CSV paket.

In [ ]:
G = nx.Graph(kind="ddos")
G.add_node("s1", type="switch")
G.add_node("srv", type="host", ip="10.0.0.100", mac="00:00:00:00:01:00")
G.add_edge("srv", "s1", delay=1.0, bandwidth=5.0, loss=0.0)        # link server sempit (5 Mbps)
for i in range(1, 7):
    G.add_node(f"h{i}", type="host", ip=f"10.0.0.{i}", mac=f"00:00:00:00:00:{i:02x}")
    G.add_edge(f"h{i}", "s1", delay=1.0, bandwidth=100.0, loss=0.0)

net = build(G, "standalone")
wait_connectivity(net)
srv = net.get("srv")
srv.cmd("mkdir -p /tmp/www && head -c 20000 /dev/urandom | base64 > /tmp/www/index.html")
srv.cmd(f"cd /tmp/www && python3 -m http.server 80 --bind {G.nodes['srv']['ip']} > /dev/null 2>&1 &")
pcap = str(OUT / "server_capture.pcap")
srv.cmd(f"tcpdump -i srv-eth0 -w {pcap} -s 96 > /dev/null 2>&1 &")   # rekam header paket menuju/dari server
time.sleep(1)

# trafik normal: client mengunduh dari server (HTTP) selama 15 detik
for h in [f"h{i}" for i in range(1, 7)]:
    net.get(h).cmd(f"(for r in $(seq 1 20); do curl -s -o /dev/null -m 5 http://{G.nodes['srv']['ip']}/; sleep 0.3; done) > /dev/null 2>&1 &")
time.sleep(15)
srv.cmd("pkill tcpdump"); time.sleep(1)
net.stop()
print("pcap tersimpan:", pcap)

In [ ]:
def pcap_to_packets(pcap_path, server_ip="10.0.0.100"):
    """Ubah pcap menjadi tabel paket (ts, src, dst, proto, len) dengan tcpdump -r (tanpa tshark)."""
    out = subprocess.run(["tcpdump", "-nn", "-tt", "-r", pcap_path], capture_output=True, text=True).stdout
    rows = []
    for line in out.splitlines():
        m = re.match(r"(\d+\.\d+) (IP6?) (\S+) > (\S+): (\w+)", line)
        if not m:
            continue
        ts, _, s, d, proto = m.groups()
        lm = re.search(r"length (\d+)", line)
        sip = s.rsplit(".", 1)[0]; dip = d.rsplit(".", 1)[0]
        rows.append((float(ts), sip, dip, proto, int(lm.group(1)) if lm else 0))
    return pd.DataFrame(rows, columns=["ts", "src", "dst", "proto", "length"])

def packet_features(pkts, server_ip="10.0.0.100", window=1.0):
    """Fitur per jendela `window` detik untuk trafik MENUJU server (gaya fitur deteksi DDoS)."""
    s = pkts[pkts.dst == server_ip].copy()
    if s.empty:
        return pd.DataFrame()
    s["win"] = ((s.ts - s.ts.min()) // window).astype(int)
    def entropy(w):
        p = w.value_counts(normalize=True)
        return float(-(p * np.log2(p)).sum()) if len(p) else 0.0
    rows = []
    for w, g in s.groupby("win"):
        rows.append(dict(win=w, pkt_rate=len(g) / window, byte_rate=g.length.sum() / window,
                         n_src_ip=g.src.nunique(), src_entropy=entropy(g.src),
                         avg_len=g.length.mean(), tcp_share=(g.proto == "tcp").mean(),
                         udp_share=(g.proto == "UDP").mean(), icmp_share=(g.proto == "ICMP").mean()))
    return pd.DataFrame(rows).set_index("win")

pkts = pcap_to_packets(pcap)
pkts.to_csv(OUT / "paket_server.csv", index=False)
feat = packet_features(pkts)
feat.to_csv(OUT / "fitur_ddos_per_jendela.csv")
print(f"{len(pkts)} paket terekam; {len(feat)} jendela fitur")
display(feat.round(2).head())

if len(feat):
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(feat.index, feat.pkt_rate, "-o", color="#2f6db5", label="paket/s ke server")
    ax.set_xlabel("jendela (s)"); ax.set_ylabel("paket/s"); ax.legend(fontsize=8)
    ax.set_title("Laju paket menuju server (trafik normal)")
    plt.tight_layout(); plt.savefig(OUT / "laju_paket_server.png", dpi=150); plt.show()

### Menyambung ke notebook deteksi DDoS
- Untuk **dataset berlabel lengkap** (normal + SYN/UDP/ICMP flood + flash crowd), jalankan `02_deteksi_ddos.ipynb` dalam mode `mininet`; notebook itu sudah membangkitkan trafik serangan dengan `hping3` **di dalam jaringan emulasi yang terisolasi** dan menyusun fitur per jendela beserta labelnya.
- Fungsi `pcap_to_packets` dan `packet_features` di atas dapat dipakai untuk menambahkan fitur berbasis `pcap` pada pipeline notebook 02.

## 7. Penutup
File hasil di `results/tutorial_mininet/` dan topologi di `data/`. Bila kernel sempat terhenti, jalankan `sudo mn -c` di terminal untuk membersihkan sisa interface.

In [ ]:
cleanup()
for f in sorted(OUT.iterdir()):
    print(f"  {f.relative_to(BASE)}  ({f.stat().st_size/1024:.1f} KB)")

### Ide eksplorasi
1. Ganti `PARAMS['fat-tree']` ke `k=6`, amati jumlah switch dan jalur ganda yang tersedia.
2. Tambah topologi dari generator acak proyek optimasi, ekspor JSON-nya, lalu bandingkan RTT terukur vs estimasi analitik di `01_optimasi_jaringan.ipynb`.
3. Pada uji link putus, bandingkan waktu pemulihan RSTP untuk ring, mesh, dan fat-tree.